# SSL-GMMVC — end-to-end voice conversion demo

This notebook walks the full pipeline on the CMU ARCTIC corpus:

1. **Feature extraction** — WavLM features for a source and a target speaker (`features.FeatureExtractor`).
2. **k-NN matching** — pair source frames with content-similar target frames (`features.symmetric_matching`).
3. **GMM training & saving** — fit a joint GMM on the aligned `[source | target]` pairs and persist it (`gmm`).
4. **Loading** — restore the model from disk.
5. **Conversion** — map a held-out source utterance into the target voice space.
6. **Vocoding** — synthesize a waveform from the converted features (`vocoder.Vocoder`).

Everything follows a single `DEVICE` choice: CUDA when available, otherwise CPU.

> **Requirements.** First run downloads the WavLM and HiFi-GAN checkpoints from
> `torch.hub` (`bshall/knn-vc`), so it needs network access. The GPU presets and
> the vocoder use CUDA when `DEVICE == "cuda"`; on CPU everything still runs,
> just slower.

## 0. Setup

In [1]:
%cd ssl-gmmvc

/home/tanabu/ssl-gmmvc


/home/tanabu/ssl-gmmvc/.venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [13]:
import torch
from pathlib import Path

from features import FeatureExtractor, symmetric_matching
from gmm import CrossDiagJointGMMGPU, CrossDiagJointGMMCPU
from vocoder import Vocoder

# CUDA when available, else CPU. Every stage below follows this choice.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# --- hyper-parameters -------------------------------------------------------
WAVLM_LAYER = 6        # WavLM layer to read features from (knn-vc default)
N_COMPONENTS = 256      # joint-GMM mixture components
K_NEIGHBOURS = 4       # neighbours per frame for symmetric k-NN matching
N_TRAIN_UTTS = 20      # utterances pooled per speaker when fitting
CHUNK_SIZE = 2_000    # E-step rows per chunk during the GMM fit (caps peak memory)

# --- artefact paths ---------------------------------------------------------
MODEL_PATH = "models/bdl_to_slt_crossdiag.npz"
OUTPUT_WAV = "outputs/bdl_to_slt.wav"

Using device: cuda


## 1. Feature extraction

Pool WavLM frames for the source and target speakers, and grab one held-out
source utterance to convert at the end. CMU ARCTIC audio is 16 kHz mono, which
is exactly what `FeatureExtractor` expects.

In [14]:
CORPUS = Path("corpus/CMU-ARCTIC")

def speaker_wavs(speaker: str) -> list[Path]:
    """Sorted wav paths for an ARCTIC speaker (16 kHz mono)."""
    return sorted((CORPUS / f"cmu_us_{speaker}_arctic" / "wav").glob("*.wav"))

SOURCE_SPEAKER = "bdl"   # US male
TARGET_SPEAKER = "slt"   # US female

source_wavs = speaker_wavs(SOURCE_SPEAKER)
target_wavs = speaker_wavs(TARGET_SPEAKER)

source_train = source_wavs[:N_TRAIN_UTTS]
target_train = target_wavs[:N_TRAIN_UTTS]
source_to_convert = source_wavs[N_TRAIN_UTTS]   # held out from training

print(f"source train: {len(source_train)} utts | target train: {len(target_train)} utts")
print(f"converting:   {source_to_convert.name}")

source train: 20 utts | target train: 20 utts
converting:   arctic_a0021.wav


In [15]:
extractor = FeatureExtractor(layer_num=WAVLM_LAYER, device=DEVICE)

# Pool source- and target-speaker frames; VAD trims leading/trailing silence.
source_features = extractor.get_features_from_file_list(source_train, vad=True)
target_features = extractor.get_features_from_file_list(target_train, vad=True)

# The single utterance we'll convert at the end.
convert_features = extractor.get_features_from_filepath(source_to_convert, vad=True)

print("source :", tuple(source_features.shape))
print("target :", tuple(target_features.shape))
print("convert:", tuple(convert_features.shape))

Using cache found in /home/tanabu/.cache/torch/hub/bshall_knn-vc_master
/home/tanabu/ssl-gmmvc/.venv/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


WavLM-Large loaded with 315,453,120 parameters.


/home/tanabu/ssl-gmmvc/.venv/lib/python3.10/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/home/tanabu/ssl-gmmvc/.venv/lib/python3.10/site-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/390

Extracted features from 20 files -> (2371, 1024) on cuda
Extracted features from 20 files -> (2320, 1024) on cuda
source : (2371, 1024)
target : (2320, 1024)
convert: (94, 1024)


## 2. k-NN matching

The joint GMM needs aligned `(source, target)` frame pairs. `symmetric_matching`
pairs each frame with its cosine-nearest neighbours in **both** directions and
dedupes, so the alignment is by acoustic content rather than by time. We then
concatenate the matched pairs into the `[source | target]` matrix the GMM fits on.

In [16]:
source_matched, target_matched = symmetric_matching(
    source_features, target_features, k=K_NEIGHBOURS
)
XY = torch.cat([source_matched, target_matched], dim=1)

print("matched pairs           :", source_matched.shape[0])
print("joint training matrix XY:", tuple(XY.shape))   # (n_pairs, 2 * feature_dim)

matched pairs           : 13665
joint training matrix XY: (13665, 2048)


## 3. GMM training & saving

A **cross-diagonal** joint covariance models the source↔target cross-covariance
while staying cheap on 1024-dim WavLM features (a full 2048×2048 joint covariance
per component would be far heavier). The GPU preset is used on CUDA, the CPU
preset otherwise.

The E-step is the memory bottleneck: forming per-frame, per-component terms
scales with the number of frames. Passing `chunk_size` to `fit` processes those
frames in row-chunks so **peak memory scales with the chunk, not the dataset** —
the result is identical (every frame is used every iteration; this is exact EM,
not a stochastic minibatch approximation). The helper below tracks the peak
memory so the effect is visible.

### A small memory tracker

`MemoryTracker` is a `with`-block context manager that reports peak GPU
allocation and peak process RSS for the code it wraps. We use it to watch the
training memory and, below, to compare chunk sizes.

In [17]:
import gc
import threading
import time


def _rss_gb():
    """Resident set size of this process, in GB (Linux /proc)."""
    with open("/proc/self/statm") as f:
        resident_pages = int(f.read().split()[1])
    return resident_pages * 4096 / 1e9


class MemoryTracker:
    """Track peak GPU + process-RSS memory (and wall time) over a ``with`` block.

    Peak GPU comes from the CUDA allocator; peak RSS is sampled on a background
    thread so it captures the transient high-water mark, not just the endpoints.
    """

    def __init__(self, device, label="fit"):
        self.device, self.label = device, label
        self.peak_gpu_gb = self.peak_rss_gb = 0.0

    def __enter__(self):
        gc.collect()
        self._stop = False
        self._rss_peak = _rss_gb()

        def sample():
            while not self._stop:
                self._rss_peak = max(self._rss_peak, _rss_gb())
                time.sleep(0.01)

        self._thread = threading.Thread(target=sample, daemon=True)
        self._thread.start()
        if self.device == "cuda":
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
        self._t0 = time.perf_counter()
        return self

    def __exit__(self, *exc):
        dt = time.perf_counter() - self._t0
        self._stop = True
        self._thread.join()
        if self.device == "cuda":
            torch.cuda.synchronize()
            self.peak_gpu_gb = torch.cuda.max_memory_allocated() / 1e9
        self.peak_rss_gb = self._rss_peak
        gpu = f" | peak GPU {self.peak_gpu_gb:5.2f} GB" if self.device == "cuda" else ""
        print(f"[{self.label:>12}] wall {dt:5.1f}s | peak RSS {self.peak_rss_gb:5.2f} GB{gpu}")
        return False

In [18]:
GMM = CrossDiagJointGMMGPU if DEVICE == "cuda" else CrossDiagJointGMMCPU
gmm = GMM(n_components=N_COMPONENTS, verbose=1)

# chunk_size caps the E-step memory; the fit stays exact (every frame, every step).
with MemoryTracker(DEVICE, "chunked fit"):
    gmm.fit(XY, max_iter=100, tol=1e-6, chunk_size=CHUNK_SIZE)

gmm.save_model(MODEL_PATH)

Starting CrossDiagJointGMMGPU fitting with 256 components
Data: 13665 samples, chunk_size=2000
Backend: torch
--------------------------------------------------
Iteration   1 | Log-likelihood/sample: -3986.697731 | Rise rate: N/A (first iteration)
Iteration  10 | Log-likelihood/sample: -3878.416941 | Rise rate:   0.00029353
Iteration  20 | Log-likelihood/sample: -3874.446286 | Rise rate:   0.00002300

✓ Converged at iteration 30 (rise rate 9.45e-09 < 1.00e-06)
--------------------------------------------------
✓ Training completed successfully!
   Final log-likelihood/sample: -3873.940432
[ chunked fit] wall  19.5s | peak RSS  4.45 GB | peak GPU 10.02 GB
Directory 'models' ensured.
Model saved to models/bdl_to_slt_crossdiag.npz


### Peak memory vs chunk size

Re-fit the same data with a few `chunk_size` values to see the memory ceiling
drop while the result stays the same. Smaller chunks trade a little speed for a
lower peak — on a bigger corpus this is the difference between fitting and an
out-of-memory error.

In [19]:
# Same data and result at every chunk size -- only the memory ceiling changes.
# chunk_size=None is the classic full-batch E-step; a finite chunk_size bounds it.
# Short max_iter here: we're measuring the peak, not training to convergence.
for cs in (None, CHUNK_SIZE, 2_000):
    probe = GMM(n_components=N_COMPONENTS, verbose=0)
    label = "whole-batch" if cs is None else f"chunk={cs}"
    with MemoryTracker(DEVICE, label):
        probe.fit(XY, max_iter=20, tol=1e-6, chunk_size=cs)

[ whole-batch] wall   1.3s | peak RSS  4.25 GB | peak GPU 44.63 GB


RuntimeError: GMM training failed due to invalid learning

## 4. Loading

Restore into a fresh instance. `load_model` reads `n_components` / `feature_dim`
back from the file, so this is independent of the object that trained it.

In [9]:
GMM = CrossDiagJointGMMGPU if DEVICE == "cuda" else CrossDiagJointGMMCPU
gmm_loaded = GMM(n_components=N_COMPONENTS, verbose=1)
gmm_loaded.load_model(MODEL_PATH)

Model loaded from models/bdl_to_slt_crossdiag.npz


## 5. Conversion

Map the held-out source utterance into target space (forward source → target).
`convert` returns the backend's native array — numpy on CPU, a tensor on CUDA —
so we coerce it to a float tensor for the vocoder.

In [10]:
converted = gmm_loaded.convert(convert_features)
converted = torch.as_tensor(converted, dtype=torch.float32)   # (n_frames, feature_dim)

print("converted features:", tuple(converted.shape))

converted features: (94, 1024)


## 6. Vocoding

Synthesize the converted features into a 16 kHz waveform with the prematched
WavLM HiFi-GAN. We also resynthesize the original source features as an A/B
reference.

In [11]:
voc = Vocoder(device=DEVICE)

converted_wav = voc.vocode(converted, output_file=OUTPUT_WAV)
source_wav = voc.vocode(convert_features)   # original speaker, for comparison

print("converted waveform:", tuple(converted_wav.shape))

Using cache found in /home/tanabu/.cache/torch/hub/bshall_knn-vc_master


Removing weight norm...
[HiFiGAN] Generator loaded with 16,523,393 parameters.
Vocoder output saved to outputs/bdl_to_slt.wav
converted waveform: (1, 30080)


/home/tanabu/ssl-gmmvc/.venv/lib/python3.10/site-packages/torchaudio/_backend/utils.py:337: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.save_with_torchcodec` under the hood. Some parameters like format, encoding, bits_per_sample, buffer_size, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's encoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.encoders.AudioEncoder
  warnings.warn(
/home/tanabu/ssl-gmmvc/.venv/lib/python3.10/site-packages/torchaudio/_backend/ffmpeg.py:247: UserWarning: torio.io._streaming_media_encoder.StreamingMediaEncoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be

In [12]:
from IPython.display import Audio, display

print(f"Source — original {SOURCE_SPEAKER} (resynthesized):")
display(Audio(source_wav.squeeze().numpy(), rate=voc.sample_rate))

print(f"Converted — {SOURCE_SPEAKER} -> {TARGET_SPEAKER}:")
display(Audio(converted_wav.squeeze().numpy(), rate=voc.sample_rate))

Source — original bdl (resynthesized):


Converted — bdl -> slt:
